# Turkish Morphology-Aware Dense Retrieval Fine-Tuning

Bu notebook final train JSONL ile encoder fine-tuning yapar. Pilot veriyle veya sealed test üzerinde model seçimiyle çalışmayı özellikle reddeder.

**Protokol:**
1. Final train → deterministik `%90 train / %10 development`.
2. Train: query + positive + iki morfolojik hard + bir semantik hard.
3. Development: Recall@1/3, MRR ve morfolojik pairwise accuracy.
4. Sealed 600: yalnız hiperparametreler dondurulduktan sonra `RUN_SEALED_TEST=True` ile bir kez.
5. Paper sonucu için aynı ayarlar en az üç seed ile tekrarlanır.


## 1. Colab kurulumu


In [ ]:
# Colab'ın önceden kurulu torchvision paketi bazı torch sürümleriyle NMS çakışması yapıyor.
# Bu text-only eğitimde torchvision ve eski torchao gerekmediği için kaldırıyoruz.
!pip uninstall -y -q torchvision torchao
!pip install -q -U "sentence-transformers>=3.1.0" "transformers>=4.48.0" peft accelerate datasets pandas

import hashlib, json, os, random, shutil, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch

print("torch", torch.__version__)
print("GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK")
assert torch.cuda.is_available(), "GPU runtime seçin."


## 2. Repo ve deney ayarları


In [ ]:
REPO_URL = "https://github.com/TR-morph-retrieval/turkish-morph-retrieval.git"
ROOT = Path("/content/turkish-morph-retrieval")
if not (ROOT / ".git").exists():
    if ROOT.exists(): shutil.rmtree(ROOT)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

MODELS = {
    "modernbert-tr": {"repo":"ytu-ce-cosmos/modernbert-tr-embed", "q":"", "d":"",
                      "targets":["Wqkv","Wo","Wi"]},
    "bge-m3": {"repo":"BAAI/bge-m3", "q":"", "d":"",
               "targets":["query","key","value","dense"]},
    "e5-large": {"repo":"intfloat/multilingual-e5-large", "q":"query: ", "d":"passage: ",
                 "targets":["query","key","value","dense"]},
    "trmteb-ft": {"repo":"trmteb/turkish-embedding-model-fine-tuned", "q":"", "d":"",
                  "targets":["query","key","value","dense"]},
}
SELECTED_KEY = "e5-large"
CFG = MODELS[SELECTED_KEY]
SEED = 42
# Pilot LoRA denemesi: yalnız güncel şemayı geçen aileler kullanılır; sealed test kapalı kalır.
PILOT_MODE = True
TRAIN_JSONL = (ROOT / "train/data/pilot/pilot60.jsonl") if PILOT_MODE else (ROOT / "train/data/final/train1000.jsonl")
RUN_SEALED_TEST = False  # yalnız bütün seçimler dondurulduğunda True
PARAMS = dict(epochs=3, batch_size=32, cache_mini_batch=16, learning_rate=1e-4,
              warmup_ratio=.1, weight_decay=.01, lora_r=16, lora_alpha=32,
              lora_dropout=.05, max_seq_length=256,
              output_dir=f"/content/{SELECTED_KEY}_morph_seed{SEED}")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(SELECTED_KEY, CFG["repo"], PARAMS)


## 3. Final train verisini doğrula ve development ayır


In [ ]:
from datasets import Dataset

def stable_dev(family_id, seed=SEED):
    h = hashlib.sha256(f"{seed}:{family_id}".encode()).digest()
    return int.from_bytes(h[:4], "big") % 10 == 0

def load_rows(path, pilot_mode):
    if not path.exists():
        raise FileNotFoundError(f"Train JSONL bulunamadı: {path}")
    raw = [json.loads(x) for x in path.read_text(encoding="utf-8").splitlines() if x.strip()]
    if not raw: raise ValueError("Train JSONL boş")
    if pilot_mode:
        from train.production import validate_family
        kept, skipped = [], []
        for record in raw:
            errors = validate_family(record["family"])
            (skipped if errors else kept).append((record, errors))
        raw = [record for record, _ in kept]
        print(f"Pilot filtresi: {len(raw)} kullanılacak, {len(skipped)} eski/uyumsuz kayıt atlandı.")
    else:
        bad = [r.get("slot_id") for r in raw
               if r.get("purpose") == "pilot_only" or r.get("eligible_for_final_train") is not True]
        if bad: raise ValueError(f"Pilot/onaysız kayıt final train'e karışmış: {bad[:5]}")
    rows=[]
    for r in raw:
        f=r["family"]; by={c["slot"]:c["text"].strip() for c in f["candidates"]}
        required={"positive","morph_1","morph_2","semantic_1"}
        if set(by) != required: raise ValueError(f"Aday şeması hatalı: {f['family_id']}")
        rows.append(dict(family_id=f["family_id"], target_feature=f["target_feature"],
                         query=f["query"].strip(), positive=by["positive"],
                         morph_1=by["morph_1"], morph_2=by["morph_2"],
                         semantic_1=by["semantic_1"]))
    ids=[r["family_id"] for r in rows]
    if len(ids)!=len(set(ids)): raise ValueError("Tekrarlı family_id")
    return rows

ALL_ROWS=load_rows(TRAIN_JSONL, PILOT_MODE)
DEV_ROWS=[r for r in ALL_ROWS if stable_dev(r["family_id"])]
TRAIN_ROWS=[r for r in ALL_ROWS if not stable_dev(r["family_id"])]
assert TRAIN_ROWS and DEV_ROWS
assert not ({r['family_id'] for r in TRAIN_ROWS} & {r['family_id'] for r in DEV_ROWS})
print(f"{'pilot' if PILOT_MODE else 'final'}={len(ALL_ROWS)} train={len(TRAIN_ROWS)} dev={len(DEV_ROWS)}")
display(pd.Series([r['target_feature'] for r in ALL_ROWS]).value_counts().describe().to_frame('fenomen dağılımı'))


## 4. Model-doğru prefix ve eğitim dataset’i


In [ ]:
def qtext(x): return CFG["q"] + x
def dtext(x): return CFG["d"] + x

# Çift yönlü veri çoğaltma (Symmetric Augmentation):
# Her aileden hem (Query -> Positive) hem (Positive -> Query) yönünde iki eğitim çifti üretilir.
train_samples = []
for r in TRAIN_ROWS:
    # 1. Asıl yön: query -> positive
    train_samples.append({
        "anchor": qtext(r["query"]), "positive": dtext(r["positive"]),
        "negative_1": dtext(r["morph_1"]), "negative_2": dtext(r["morph_2"]),
        "negative_3": dtext(r["semantic_1"])
    })
    # 2. Simetrik ters yön: positive -> query
    train_samples.append({
        "anchor": qtext(r["positive"]), "positive": dtext(r["query"]),
        "negative_1": dtext(r["morph_1"]), "negative_2": dtext(r["morph_2"]),
        "negative_3": dtext(r["semantic_1"])
    })

train_dataset = Dataset.from_list(train_samples)
print(f"Toplam eğitim örneği (simetrik çoğaltma ile): {len(train_dataset)} (Kaynak aile sayısı: {len(TRAIN_ROWS)})")
print(train_dataset[0])


## 5. Development retrieval metriği


In [ ]:
def rank_dev(model, rows, batch_size=32):
    queries=[qtext(r["query"]) for r in rows]
    docs=[]
    for r in rows: docs += [dtext(r[k]) for k in ("positive","morph_1","morph_2","semantic_1")]
    qe=model.encode(queries,batch_size=batch_size,normalize_embeddings=True,convert_to_numpy=True)
    de=model.encode(docs,batch_size=batch_size,normalize_embeddings=True,convert_to_numpy=True).reshape(len(rows),4,-1)
    scores=np.einsum("bd,bkd->bk",qe,de)
    ranks=np.array([1+np.sum(s[1:]>s[0]) for s in scores])
    return {
        "families":len(rows), "recall@1":float(np.mean(ranks<=1)),
        "recall@3":float(np.mean(ranks<=3)), "mrr":float(np.mean(1/ranks)),
        "pairwise_morph_accuracy":float(np.mean(scores[:,0,None] > scores[:,1:3])),
        "family_consistency":float(np.mean(np.all(scores[:,0,None] > scores[:,1:],axis=1))),
        "mean_rank":float(np.mean(ranks))}


## 6. Taban modeli yükle ve dev başlangıcını ölç


In [ ]:
from sentence_transformers import SentenceTransformer

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model=SentenceTransformer(CFG["repo"],trust_remote_code=True,
                          model_kwargs={"torch_dtype":dtype})
model.max_seq_length=PARAMS["max_seq_length"]
BASE_DEV=rank_dev(model,DEV_ROWS)
display(pd.DataFrame([{"stage":"zero-shot",**BASE_DEV}]))


## 7. LoRA’yı doğru modüle bağla ve cached contrastive loss kur


In [ ]:
from peft import LoraConfig, get_peft_model
from sentence_transformers.sentence_transformer.losses import CachedMultipleNegativesRankingLoss

first=model._first_module()
available={name.rsplit('.',1)[-1] for name, _ in first.auto_model.named_modules()}
targets=[name for name in CFG["targets"] if name in available]
if not targets: raise RuntimeError(f"LoRA hedefi bulunamadı. Kullanılabilir adlar: {sorted(available)[:40]}")
print("LoRA targets:", targets)
peft_model=get_peft_model(first.auto_model,LoraConfig(
    r=PARAMS["lora_r"], lora_alpha=PARAMS["lora_alpha"],
    lora_dropout=PARAMS["lora_dropout"], bias="none",
    target_modules=targets, task_type="FEATURE_EXTRACTION"))
first.auto_model=peft_model  # kritik: wrapper SentenceTransformer'a geri bağlanır
peft_model.print_trainable_parameters()
loss=CachedMultipleNegativesRankingLoss(model,scale=20.0,
                                         mini_batch_size=PARAMS["cache_mini_batch"])
print("Cached MNRL: üç açık hard negatif + büyük etkili batch")


## 8. Eğit


In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers

args=SentenceTransformerTrainingArguments(
    output_dir=PARAMS["output_dir"], num_train_epochs=PARAMS["epochs"],
    per_device_train_batch_size=PARAMS["batch_size"],
    learning_rate=PARAMS["learning_rate"], warmup_ratio=PARAMS["warmup_ratio"],
    weight_decay=PARAMS["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    batch_sampler=BatchSamplers.NO_DUPLICATES, logging_steps=10,
    save_strategy="epoch", seed=SEED, data_seed=SEED, report_to="none")
trainer=SentenceTransformerTrainer(model=model,args=args,train_dataset=train_dataset,loss=loss)
result=trainer.train()
print(result)


## 9. Development sonrası karşılaştırma


In [ ]:
FT_DEV=rank_dev(model,DEV_ROWS)
comparison=pd.DataFrame([
    {"stage":"zero-shot",**BASE_DEV},
    {"stage":"fine-tuned",**FT_DEV},
    {"stage":"delta",**{k:(FT_DEV[k]-BASE_DEV[k]) for k in BASE_DEV if k!="families"},
     "families":len(DEV_ROWS)}])
display(comparison)


## 10. Sealed 600 — yalnız final koşuda


In [ ]:
if RUN_SEALED_TEST:
    from test.evaluation import load_items, score_encoder
    shard_dir=ROOT/"test/data/final_shards"
    sealed_tmp=Path("/content/morph_test_600_sealed.json")
    items=[json.loads(line) for sf in sorted(shard_dir.glob("*.jsonl"))
           for line in sf.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert len(items)==600 and all(x.get("target_split")=="sealed_test" for x in items)
    sealed_tmp.write_text(json.dumps({"items":items},ensure_ascii=False),encoding="utf-8")
    sealed=load_items(sealed_tmp)
    final_result=score_encoder(model,sealed,query_prefix=CFG["q"],
                               document_prefix=CFG["d"],batch_size=32)
    display(pd.DataFrame([final_result["summary"]]))
else:
    print("Sealed test açılmadı. Model/hyperparameter seçimini yalnız development ile yapın.")


## 11. Model ve deney manifestini kaydet


In [ ]:
out=Path(PARAMS["output_dir"]); save_path=out/("pilot_model" if PILOT_MODE else "final_model")
model.save(str(save_path))
manifest={"model":CFG["repo"],"selected_key":SELECTED_KEY,"seed":SEED,
          "params":PARAMS,"train_file":str(TRAIN_JSONL),
          "train_file_sha256":hashlib.sha256(TRAIN_JSONL.read_bytes()).hexdigest(),
          "train_families":len(TRAIN_ROWS),"dev_families":len(DEV_ROWS),
          "base_dev":BASE_DEV,"fine_tuned_dev":FT_DEV,
          "pilot_mode":PILOT_MODE,"sealed_test_opened":RUN_SEALED_TEST}
(out/"experiment_manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
print("Kaydedildi:",save_path)
